# Проверка качества данных

## Цель:

Оценить полноту, уникальность, согласованность и логическую
корректность данных. На основе этой информации сформулировать критерии для отчистки



In [8]:
import pandas as pd
from pathlib import Path

In [28]:
CURRENT_DIR=Path.cwd()
PROJECT_ROOT=(
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)
DATA_PATH = (PROJECT_ROOT / "data"/ "processed"/ "complaints_initialized.pkl")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Файл не найден: {DATA_PATH}")
df=pd.read_pickle(DATA_PATH)
print("Файл:", DATA_PATH)
print("Размер таблицы:", df.shape) #ожидаемый результат (141822, 19)

Файл: /Users/zemfiraramazanova/PycharmProjects/financial-complaints-radar/data/processed/complaints_initialized.pkl
Размер таблицы: (141822, 19)


Обозначения полей:
- dtype хранит тип, который имеют значения, в соответствующих полях
- missing_count хранит количество пропусков по каждому столбцу
- missing_percent хранит процент пропусков по каждому столбцу
- unique_count хранит количество уникальных значений в каждом столбце

In [29]:
quality_report = pd.DataFrame(
    {   "dtype": df.dtypes.astype("string"),
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean().mul(100).round(2)),
        "unique_count": df.nunique(dropna=True) }
)
quality_report=quality_report.sort_values("missing_percent", ascending=False)
quality_report

,dtype,missing_count,missing_percent,unique_count
Tags,str,125389,88.41,3
Company public response,str,99916,70.45,9
Sub-issue,str,63000,44.42,79
State,str,873,0.62,60
Date sent to company,"datetime64[us, UTC]",0,0.00,138209
month,int32,0,0.00,3
year,int32,0,0.00,2
source,str,0,0.00,6
Complaint ID,string,0,0.00,141822
Timely response?,str,0,0.00,2


#### Наблюдения по пропускам

- Большая доля пропусков в Tags является ожидаемой, потому что специальные теги присваиваются только определенным жалобам
- Большая доля пропусков в Company public response также является ожидаемой, поскольку публичный комментарий компании является необязательным
- Аналогично для Sub-issue, для проблемы необязательно иметь уточнение проблемы
- Пропуски в State ограничивают географический анализ
- В одной строке отсутствуют Sub-product, Issue и Sub-issue.
  Эта неполная запись требует отдельного рассмотрения.

#### Рассмотрим строку без Issue

In [30]:
missing_issue_str=(df["Issue"].isna()
    | df["Sub-product"].isna()
)
df.loc[
    missing_issue_str,
    [
        "Complaint ID",
        "Date received",
        "Product",
        "Sub-product",
        "Issue",
        "Sub-issue",
        "Company",
        "source",
    ]]

,Complaint ID,Date received,Product,Sub-product,Issue,Sub-issue,Company,source
25506,8550796,2024-03-13 09:05:24+00:00,Checking or savings account,NaN,NaN,NaN,WELLS FARGO & COMPANY,complaints-2024-03.csv


#### Решение по неполной записи

В жалобе с Complaint ID = 8550796 одновременно отсутствуют
Sub-product, Issue и Sub-issue.

Остальные ключевые поля заполнены: нам известны продукт, компания,
дата и идентификатор жалобы. Поэтому строка не удаляется из общего
набора данных.

Следует учитывать ее в подсчёте общего количества жалоб,
анализе продуктов и компаний, но исключить из расчётов,
для которых требуется поле Issue.

#### Проверка дубликатов

In [31]:
duplicate_check=(df["Complaint ID"].duplicated(keep=False))
duplicate_rows= (df.loc[duplicate_check].sort_values("Complaint ID"))
print( "Строк с повторяющимся ID:", len(duplicate_rows))

Строк с повторяющимся ID: 0


Соответственно, при отчистке удаление дубликатов не требуется

#### Проверка соотношения значений по категориям

In [33]:
expected={
    "Product": {
        "Checking or savings account",
        "Credit card",
        "Money transfer, virtual currency, or money service"
    },
    "Timely response?": {"Yes", "No"},
    "Company response to consumer": {
        "Closed with explanation",
        "Closed with non-monetary relief",
        "Closed with monetary relief",
        "Untimely response"
    },
    "Submitted via": {
        "Web",
        "Phone",
        "Referral",
        "Postal mail"
    }
}
category_check=[]
for column, expected_values in expected.items():
    cur=set(df[column].dropna())
    category_check.append(
        {
            "column": column,
            "missing_count": (df[column].isna().sum()),
            "unexpected_values": sorted(cur-expected_values),
            "missing_expected_values": sorted(expected_values - cur)
        }
    )
category_check=pd.DataFrame(category_check)
category_check


,column,missing_count,unexpected_values,missing_expected_values
0,Product,0,[],[]
1,Timely response?,0,[],[]
2,Company response to consumer,0,[],[]
3,Submitted via,0,[],[]


### Результат проверки категорий

В ключевых категориальных столбцах отсутствуют пропуски и
неожиданные значения.

`Timely response?` и `Company response to consumer` описывают
разные признаки. Первое поле показывает соблюдение срока ответа,
а второе показывает тип ответа компании. Их не надо объединять
или воспринимать взаимозаменяемыми

#### Проверка некорректных временных промежутков

In [36]:
sent_before_received=(df["Date sent to company"]<df["Date received"])
print("Количество писем, отправленных раньше получения равно", sent_before_received.sum())

Количество писем, отправленных раньше получения равно 0


Ошибок связанных со временем нет, отчистка не требуется

## Итоги проверки качества

1. Таблица содержит 141 822 строки;
2. Все значения `Complaint ID` заполнены и уникальны;
3. Ключевые поля `Date received`, `Product`, `Company` и `Complaint ID` заполнены полностью;
4. Одна жалоба не содержит `Sub-product`, `Issue` и `Sub-issue`.
   Она сохраняется в общем наборе, но не будет участвовать в
   анализе проблем
5. Пропуски в `Tags`, `Company public response` и `Sub-issue`
   обусловлены структурой данных и не требуют заполнения.
6. Строки без `State` сохраняются, но не используются для географического анализа;
7. Время и даты заполнены правильно, нет конфликтов по времени;
## Решения для очистки

- Переименовать столбцы в стиле `snake_case` для упрощения чтения кода;
- Не удалять строку без `Issue` из общего набора.
- Не удалять строки без `State` из общего набора.
- Преобразовать `Timely response?` в логический признак.
